# TabPFN 5-fold false-positive mining (VLST)

Runs TabPFN five times with an outer **StratifiedKFold** split. The five test folds are disjoint and their union is the original dataset. Inside each fold, the non-test rows are split into train + calibration, so each fold has disjoint train/calibration/test rows.

Outputs are TabPFN-only false positives from the fold test sets:

- all thresholds on `ALL_THRESHOLDS` (`0.05..0.95`, step `0.01` by default)
- focused thresholds on `FOCUS_THRESHOLDS` (`0.20..0.60`, step `0.01` by default)
- unique FP datasets with no repeated `full_row_id`
- long FP datasets with one row per `(full_row_id, threshold)` pair
- per-threshold counts and fold summaries

Note: standard 5-fold CV has overlapping training sets across folds. What is guaranteed here is: no test-fold overlap, test-fold union equals the full dataset, and train/calibration/test are disjoint within each fold.


## 1. Install TabPFN client

Run this once per fresh environment. This matches `tabpfn_fp_followup.ipynb`: use **`tabpfn-client`** cloud API with an explicit token setup cell, so the client never opens an interactive login prompt.


In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "tabpfn-client"])
print("Installed/updated tabpfn-client")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.4/279.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 24.0.0
    Uninstalling pyarrow-24.0.0:
      Successfully uninstalled pyarrow-24.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.2

Installed/updated tabpfn-client


## 2. Imports and configuration

Set `VLST_RAW_CSV`, `VLST_5FOLD_FP_OUT_DIR`, or `TABPFN_TOKEN` in the environment if auto-detection is not enough. On Kaggle, you can also store the key as secret `TABPFN_TOKEN_H`, matching `tabpfn_fp_followup.ipynb`.


In [2]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

TARGET_COL = "Stent thrombosis"
DROP_FEATURES = ["Time since stent implantation"]
ID_COLS = ["NO.", "Name"]
RANDOM_STATE = int(os.environ.get("VLST_5FOLD_RANDOM_STATE", "42"))
N_SPLITS = int(os.environ.get("VLST_5FOLD_N_SPLITS", "5"))
CAL_SIZE = float(os.environ.get("VLST_5FOLD_CAL_SIZE", "0.15"))

ALL_THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.01), 2)
FOCUS_THRESHOLDS = np.round(np.arange(0.20, 0.61, 0.01), 2)

TABPFN_N_ESTIMATORS = int(os.environ.get("TABPFN_N_ESTIMATORS", os.environ.get("VLST_TABPFN_N_ESTIMATORS", "8")))
BALANCE_PROBABILITIES = os.environ.get("VLST_TABPFN_BALANCE_PROBABILITIES", "1").strip().lower() not in {"0", "false", "no"}
IGNORE_PRETRAINING_LIMITS = os.environ.get("VLST_TABPFN_IGNORE_PRETRAINING_LIMITS", "1").strip().lower() not in {"0", "false", "no"}

print("N_SPLITS:", N_SPLITS)
print("CAL_SIZE:", CAL_SIZE)
print("ALL_THRESHOLDS:", f"{ALL_THRESHOLDS[0]:.2f}..{ALL_THRESHOLDS[-1]:.2f}", "n=", len(ALL_THRESHOLDS))
print("FOCUS_THRESHOLDS:", f"{FOCUS_THRESHOLDS[0]:.2f}..{FOCUS_THRESHOLDS[-1]:.2f}", "n=", len(FOCUS_THRESHOLDS))
print("TabPFN backend: tabpfn-client (cloud API)")
print("TABPFN_N_ESTIMATORS:", TABPFN_N_ESTIMATORS)


N_SPLITS: 5
CAL_SIZE: 0.15
ALL_THRESHOLDS: 0.05..0.95 n= 91
FOCUS_THRESHOLDS: 0.20..0.60 n= 41
TabPFN backend: tabpfn-client (cloud API)
TABPFN_N_ESTIMATORS: 8


## 3. Resolve paths and load VLST

The raw loader mirrors `tabpfn.ipynb`: it drops ID columns, drops the leakage feature, keeps NaNs, and integer-codes text columns without one-hot encoding.


In [3]:
def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "data").exists() and (cand / "code").exists():
            return cand
    return p


def _discover_vlst_csv() -> Path:
    env = os.environ.get("VLST_RAW_CSV") or os.environ.get("VLST_FULL_DATA_PATH")
    if env:
        p = Path(env).expanduser()
        if p.is_file():
            return p
        raise FileNotFoundError(f"Configured VLST csv does not exist: {p}")

    candidates = []
    if os.path.isdir("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if "VLST.csv" in files:
                candidates.append(Path(root) / "VLST.csv")
    repo = _find_repo_root()
    candidates.append(repo / "data" / "raw" / "VLST.csv")

    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError("Could not find VLST.csv. Set VLST_RAW_CSV or VLST_FULL_DATA_PATH.")


def _resolve_out_dir() -> Path:
    env = os.environ.get("VLST_5FOLD_FP_OUT_DIR")
    if env:
        out = Path(env).expanduser()
    elif os.path.isdir("/kaggle/working"):
        out = Path("/kaggle/working/vlst_tabpfn_5fold_fp_output")
    else:
        out = _find_repo_root() / "data" / "result" / "tabpfn_5fold_fp_mining"
    out.mkdir(parents=True, exist_ok=True)
    return out


RAW_PATH = _discover_vlst_csv()
OUT_DIR = _resolve_out_dir()
print("RAW_PATH:", RAW_PATH)
print("OUT_DIR:", OUT_DIR)


def load_raw_vlst(raw_path: Path):
    df = pd.read_csv(raw_path, low_memory=False)
    original_row_id = np.arange(len(df), dtype=int)
    df_model = df.drop(columns=[c for c in ID_COLS if c in df.columns])
    y = pd.to_numeric(df_model[TARGET_COL], errors="coerce").fillna(0).astype(int).to_numpy()
    drop = [TARGET_COL] + [c for c in DROP_FEATURES if c in df_model.columns]
    X_df = df_model.drop(columns=drop).copy()
    for c in X_df.columns:
        if not pd.api.types.is_numeric_dtype(X_df[c]):
            codes, _ = pd.factorize(X_df[c], sort=True)
            codes = pd.Series(codes, index=X_df.index).replace(-1, np.nan)
            X_df[c] = codes
        X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
    return X_df.to_numpy(dtype=float), y, list(X_df.columns), original_row_id, df


X_all, y_all, feature_names, full_row_id, df_raw = load_raw_vlst(RAW_PATH)
if not set(np.unique(y_all)).issubset({0, 1}):
    raise ValueError("Target column must be binary 0/1.")
print("Loaded:", X_all.shape, "features=", len(feature_names))
print("Target counts:", dict(zip(*np.unique(y_all, return_counts=True))))


RAW_PATH: /kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv
OUT_DIR: /kaggle/working/vlst_tabpfn_5fold_fp_output
Loaded: (5185, 81) features= 81
Target counts: {np.int64(0): np.int64(5093), np.int64(1): np.int64(92)}


## 4. TabPFN helpers


In [4]:
from tabpfn_client import TabPFNClassifier, set_access_token

_KAGGLE_SECRET_NAME = "TABPFN_TOKEN_H"

_token = os.environ.get("TABPFN_TOKEN", "").strip()
if not _token:
    _token = os.environ.get("TABPFN_TOKEN_H", "").strip()
if not _token:
    try:
        from kaggle_secrets import UserSecretsClient

        _token = str(UserSecretsClient().get_secret(_KAGGLE_SECRET_NAME)).strip()
    except Exception:
        pass

if _token:
    os.environ["TABPFN_TOKEN"] = _token
    set_access_token(_token)
    print("TabPFN client: access token configured (value hidden).")
else:
    raise RuntimeError(
        f"TABPFN_TOKEN missing. Set TABPFN_TOKEN / TABPFN_TOKEN_H (env) or Kaggle secret {_KAGGLE_SECRET_NAME!r}. "
        "API key: https://ux.priorlabs.ai/account"
    )


def make_tabpfn(seed: int) -> TabPFNClassifier:
    """tabpfn-client API (cloud); no local `device` argument."""
    return TabPFNClassifier(
        random_state=int(seed),
        n_estimators=TABPFN_N_ESTIMATORS,
        ignore_pretraining_limits=bool(IGNORE_PRETRAINING_LIMITS),
        balance_probabilities=bool(BALANCE_PROBABILITIES),
    )


def positive_proba(clf, X):
    classes = list(clf.classes_)
    idx = classes.index(1) if 1 in classes else 1
    return np.asarray(clf.predict_proba(X)[:, idx], dtype=float)


def safe_train_cal_split(trainval_idx, y, cal_size, seed):
    y_tv = y[trainval_idx]
    _, counts = np.unique(y_tv, return_counts=True)
    stratify = y_tv if len(counts) == 2 and int(counts.min()) >= 2 else None
    tr, cal = train_test_split(
        trainval_idx,
        test_size=cal_size,
        random_state=seed,
        shuffle=True,
        stratify=stratify,
    )
    return np.asarray(tr, dtype=int), np.asarray(cal, dtype=int)


def threshold_list_for_score(score, thresholds):
    return [float(t) for t in thresholds if float(score) >= float(t)]


def build_long_fp_table(pred_df, thresholds, source_prefix):
    parts = []
    counts = []
    neg = pred_df["y_true"].to_numpy(dtype=int) == 0
    scores = pred_df["p_tabpfn"].to_numpy(dtype=float)
    for t in thresholds:
        t = float(t)
        m = neg & (scores >= t)
        counts.append({"threshold": t, "n_fp_test_negatives": int(m.sum())})
        if m.any():
            cols = ["fold", "full_row_id", "fold_test_position", "y_true", "p_tabpfn"]
            part = pred_df.loc[m, cols].copy()
            part.insert(2, "threshold", t)
            part["source"] = f"{source_prefix}_t_{t:.2f}"
            parts.append(part)
    df_long = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=["fold", "full_row_id", "threshold", "fold_test_position", "y_true", "p_tabpfn", "source"]
    )
    df_counts = pd.DataFrame(counts)
    return df_long, df_counts


def build_unique_fp_table(pred_df, thresholds, source, X_all, feature_names):
    min_t = float(np.min(thresholds))
    m = (pred_df["y_true"].to_numpy(dtype=int) == 0) & (pred_df["p_tabpfn"].to_numpy(dtype=float) >= min_t)
    base = pred_df.loc[m, ["fold", "full_row_id", "fold_test_position", "y_true", "p_tabpfn"]].copy()
    if base.empty:
        return pd.DataFrame(columns=["fold", "full_row_id", "fold_test_position", "y_true", "p_tabpfn"])

    if base["full_row_id"].duplicated().any():
        dupes = base.loc[base["full_row_id"].duplicated(), "full_row_id"].tolist()
        raise RuntimeError(f"Duplicate full_row_id in unique FP candidate table: {dupes[:10]}")

    caught = [threshold_list_for_score(p, thresholds) for p in base["p_tabpfn"].to_numpy(dtype=float)]
    base["n_tabpfn_thresholds"] = [len(x) for x in caught]
    base["tabpfn_threshold_min"] = [min(x) if x else np.nan for x in caught]
    base["tabpfn_threshold_max"] = [max(x) if x else np.nan for x in caught]
    base["tabpfn_thresholds"] = [";".join(f"{t:.2f}" for t in x) for x in caught]
    base["source"] = source

    feature_block = pd.DataFrame(X_all[base["full_row_id"].to_numpy(dtype=int)], columns=feature_names)
    return pd.concat([base.reset_index(drop=True), feature_block.reset_index(drop=True)], axis=1)


TabPFN client: access token configured (value hidden).


## 5. Five-fold TabPFN runs

For each fold: split the non-test rows into train/calibration, fit TabPFN on train, predict calibration and test, and collect test probabilities. Calibration predictions are saved for threshold diagnostics but false positives are mined from test folds only.


In [5]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

pred_parts = []
cal_parts = []
fold_rows = []
test_seen = []

for fold, (trainval_idx, test_idx) in enumerate(skf.split(X_all, y_all), start=1):
    fold_seed = RANDOM_STATE + fold
    train_idx, cal_idx = safe_train_cal_split(trainval_idx, y_all, CAL_SIZE, fold_seed)

    sets = {
        "train": set(map(int, train_idx)),
        "cal": set(map(int, cal_idx)),
        "test": set(map(int, test_idx)),
    }
    if sets["train"] & sets["cal"] or sets["train"] & sets["test"] or sets["cal"] & sets["test"]:
        raise RuntimeError(f"Fold {fold}: train/cal/test overlap detected")

    print(
        f"Fold {fold}/{N_SPLITS} | train={len(train_idx)} cal={len(cal_idx)} test={len(test_idx)} "
        f"| test positives={int(y_all[test_idx].sum())} negatives={int((y_all[test_idx] == 0).sum())}"
    )
    t0 = time.time()
    clf = make_tabpfn(fold_seed)
    clf.fit(X_all[train_idx], y_all[train_idx])
    p_cal = positive_proba(clf, X_all[cal_idx])
    p_test = positive_proba(clf, X_all[test_idx])
    elapsed = time.time() - t0

    pred_parts.append(
        pd.DataFrame(
            {
                "fold": fold,
                "full_row_id": full_row_id[test_idx],
                "fold_test_position": np.arange(len(test_idx), dtype=int),
                "y_true": y_all[test_idx],
                "p_tabpfn": p_test,
            }
        )
    )
    cal_parts.append(
        pd.DataFrame(
            {
                "fold": fold,
                "full_row_id": full_row_id[cal_idx],
                "y_true": y_all[cal_idx],
                "p_tabpfn": p_cal,
            }
        )
    )
    fold_rows.append(
        {
            "fold": fold,
            "n_train": int(len(train_idx)),
            "n_cal": int(len(cal_idx)),
            "n_test": int(len(test_idx)),
            "n_train_pos": int(y_all[train_idx].sum()),
            "n_cal_pos": int(y_all[cal_idx].sum()),
            "n_test_pos": int(y_all[test_idx].sum()),
            "fit_predict_seconds": float(elapsed),
        }
    )
    test_seen.extend(map(int, full_row_id[test_idx]))

pred_df = pd.concat(pred_parts, ignore_index=True)
cal_pred_df = pd.concat(cal_parts, ignore_index=True)
fold_summary = pd.DataFrame(fold_rows)

if len(test_seen) != len(set(test_seen)):
    raise RuntimeError("Outer test folds are not disjoint")
if set(test_seen) != set(map(int, full_row_id.tolist())):
    raise RuntimeError("Outer test fold union does not equal the full dataset")
if pred_df["full_row_id"].duplicated().any():
    raise RuntimeError("A row appears in more than one test fold")

print("5-fold test coverage OK:", len(test_seen), "unique rows")
print(fold_summary.to_string(index=False))


Fold 1/5 | train=3525 cal=623 test=1037 | test positives=18 negatives=1019
00:03 Fitting... Done!
00:03 Predicting... Done!
00:03 Predicting... Done!
Fold 2/5 | train=3525 cal=623 test=1037 | test positives=18 negatives=1019
00:03 Fitting... Done!
00:03 Predicting... Done!
00:03 Predicting... Done!
Fold 3/5 | train=3525 cal=623 test=1037 | test positives=18 negatives=1019
00:02 Fitting... Done!
00:03 Predicting... Done!
00:03 Predicting... Done!
Fold 4/5 | train=3525 cal=623 test=1037 | test positives=19 negatives=1018
00:02 Fitting... Done!
00:03 Predicting... Done!
00:04 Predicting... Done!
Fold 5/5 | train=3525 cal=623 test=1037 | test positives=19 negatives=1018
00:02 Fitting... Done!
00:03 Predicting... Done!
00:04 Predicting... Done!
5-fold test coverage OK: 5185 unique rows
 fold  n_train  n_cal  n_test  n_train_pos  n_cal_pos  n_test_pos  fit_predict_seconds
    1     3525    623    1037           63         11          18            11.787325
    2     3525    623    1037     

## 6. Gather false positives by threshold

The long tables intentionally have one row per `(record, threshold)` pair. The unique tables have one row per `full_row_id` and no replicated false positives.


In [6]:
df_long_all, df_counts_all = build_long_fp_table(pred_df, ALL_THRESHOLDS, "tabpfn_5fold_all")
df_unique_all = build_unique_fp_table(
    pred_df, ALL_THRESHOLDS, "tabpfn_5fold_all_threshold_union", X_all, feature_names
)

df_long_focus, df_counts_focus = build_long_fp_table(pred_df, FOCUS_THRESHOLDS, "tabpfn_5fold_t20_60")
df_unique_focus = build_unique_fp_table(
    pred_df, FOCUS_THRESHOLDS, "tabpfn_5fold_threshold_20_60_union", X_all, feature_names
)

for name, df in [("all", df_unique_all), ("threshold_20_60", df_unique_focus)]:
    if "full_row_id" in df.columns and df["full_row_id"].duplicated().any():
        dupes = df.loc[df["full_row_id"].duplicated(), "full_row_id"].tolist()
        raise RuntimeError(f"Duplicate FP rows in {name} unique output: {dupes[:10]}")

print("All thresholds | long rows:", len(df_long_all), "| unique FP rows:", len(df_unique_all))
print("Threshold 0.20-0.60 | long rows:", len(df_long_focus), "| unique FP rows:", len(df_unique_focus))
print("\nAll-threshold FP counts:")
print(df_counts_all.to_string(index=False))
print("\nThreshold 0.20-0.60 FP counts:")
print(df_counts_focus.to_string(index=False))


All thresholds | long rows: 53722 | unique FP rows: 1839
Threshold 0.20-0.60 | long rows: 26678 | unique FP rows: 1031

All-threshold FP counts:
 threshold  n_fp_test_negatives
      0.05                 1839
      0.06                 1724
      0.07                 1628
      0.08                 1563
      0.09                 1504
      0.10                 1446
      0.11                 1380
      0.12                 1335
      0.13                 1285
      0.14                 1240
      0.15                 1203
      0.16                 1173
      0.17                 1142
      0.18                 1112
      0.19                 1077
      0.20                 1031
      0.21                 1001
      0.22                  974
      0.23                  952
      0.24                  928
      0.25                  904
      0.26                  876
      0.27                  854
      0.28                  833
      0.29                  814
      0.30             

## 7. Save artifacts


In [7]:
paths = {
    "test_predictions": OUT_DIR / "tabpfn_5fold_test_predictions.csv",
    "calibration_predictions": OUT_DIR / "tabpfn_5fold_calibration_predictions.csv",
    "fold_summary": OUT_DIR / "tabpfn_5fold_summary.csv",
    "all_counts": OUT_DIR / "tabpfn_5fold_fp_counts_by_threshold.csv",
    "all_long": OUT_DIR / "false_positives_tabpfn_5fold_all_thresholds_long.csv",
    "all_unique": OUT_DIR / "false_positives_tabpfn_5fold_test.csv",
    "focus_counts": OUT_DIR / "tabpfn_5fold_fp_counts_threshold_20_60.csv",
    "focus_long": OUT_DIR / "false_positives_tabpfn_5fold_threshold_20_60_long.csv",
    "focus_unique": OUT_DIR / "false_positives_tabpfn_5fold_threshold_20_60.csv",
    "summary_json": OUT_DIR / "tabpfn_5fold_fp_summary.json",
}

pred_df.to_csv(paths["test_predictions"], index=False)
cal_pred_df.to_csv(paths["calibration_predictions"], index=False)
fold_summary.to_csv(paths["fold_summary"], index=False)
df_counts_all.to_csv(paths["all_counts"], index=False)
df_long_all.to_csv(paths["all_long"], index=False)
df_unique_all.to_csv(paths["all_unique"], index=False)
df_counts_focus.to_csv(paths["focus_counts"], index=False)
df_long_focus.to_csv(paths["focus_long"], index=False)
df_unique_focus.to_csv(paths["focus_unique"], index=False)

summary = {
    "raw_path": str(RAW_PATH),
    "out_dir": str(OUT_DIR),
    "source_model": "tabpfn",
    "n_splits": int(N_SPLITS),
    "cal_size": float(CAL_SIZE),
    "random_state": int(RANDOM_STATE),
    "n_rows": int(len(y_all)),
    "n_features": int(len(feature_names)),
    "target_counts": {str(int(k)): int(v) for k, v in zip(*np.unique(y_all, return_counts=True))},
    "all_thresholds": [float(x) for x in ALL_THRESHOLDS],
    "focus_thresholds": [float(x) for x in FOCUS_THRESHOLDS],
    "n_long_fp_rows_all_thresholds": int(len(df_long_all)),
    "n_unique_fp_rows_all_thresholds": int(len(df_unique_all)),
    "n_long_fp_rows_threshold_20_60": int(len(df_long_focus)),
    "n_unique_fp_rows_threshold_20_60": int(len(df_unique_focus)),
    "test_fold_coverage_unique_rows": int(pred_df["full_row_id"].nunique()),
    "test_fold_coverage_is_full_dataset": bool(pred_df["full_row_id"].nunique() == len(y_all)),
    "artifacts": {k: str(v) for k, v in paths.items()},
}
with open(paths["summary_json"], "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

for label, path in paths.items():
    print("Saved", label, "->", path)


Saved test_predictions -> /kaggle/working/vlst_tabpfn_5fold_fp_output/tabpfn_5fold_test_predictions.csv
Saved calibration_predictions -> /kaggle/working/vlst_tabpfn_5fold_fp_output/tabpfn_5fold_calibration_predictions.csv
Saved fold_summary -> /kaggle/working/vlst_tabpfn_5fold_fp_output/tabpfn_5fold_summary.csv
Saved all_counts -> /kaggle/working/vlst_tabpfn_5fold_fp_output/tabpfn_5fold_fp_counts_by_threshold.csv
Saved all_long -> /kaggle/working/vlst_tabpfn_5fold_fp_output/false_positives_tabpfn_5fold_all_thresholds_long.csv
Saved all_unique -> /kaggle/working/vlst_tabpfn_5fold_fp_output/false_positives_tabpfn_5fold_test.csv
Saved focus_counts -> /kaggle/working/vlst_tabpfn_5fold_fp_output/tabpfn_5fold_fp_counts_threshold_20_60.csv
Saved focus_long -> /kaggle/working/vlst_tabpfn_5fold_fp_output/false_positives_tabpfn_5fold_threshold_20_60_long.csv
Saved focus_unique -> /kaggle/working/vlst_tabpfn_5fold_fp_output/false_positives_tabpfn_5fold_threshold_20_60.csv
Saved summary_json -> /k